# Recurrence Quantification Analysis (RQA) of Regional SPI-12 Series — Zacatecas

**Independent project** — distinct from the MF-DFA manuscript under review at *Atmosphere* and from the book chapter on AI models for drought forecasting. This notebook explores whether Recurrence Quantification Analysis (RQA) and Cross/Joint Recurrence Plots (CRP/JRP) provide information complementary to that already obtained with MF-DFA across the four physiographic regions of Zacatecas (Semi-arid, Highlands, Mountains, Canyons).

**Questions this analysis aims to answer:**
1. How predictable/deterministic is the drought dynamic in each region (%DET)?
2. How long does the system typically stay "trapped" in a regime (wet or dry) before transitioning (%LAM, TT)? This is a direct operational measure of drought persistence.
3. Are drought regimes synchronized across regions (CRP, F-synchronization measure)?
4. Does synchronization between regions change before/after documented severe drought episodes (sliding-window analysis, following the methodology of Semeraro et al. 2020, *Remote Sensing*, originally applied to Amazonian vegetation indices)?

**Main methodological references:** Marwan et al. (2007) *Phys. Rep.* 438, 237–329 (RQA); Semeraro et al. (2020) *Remote Sens.* 12, 907 (CRP/JRP + F measure + sliding windows).

---
### 0. Notes before running

- This notebook is designed for Google Colab, with no GPU dependency.
- It needs the regional SPI-12 series from the MF-DFA project (`Proyecto_AI_SPI_2026/data/mfdfa/regional_SPI12.csv`), the same series underlying the manuscript under review at *Atmosphere*. Point `DRIVE_BASE`/`INPUT_CSV` to that file in Drive, or upload the CSV manually if it still only lives on Synology.
- The notebook uses a **custom NumPy implementation of RQA** (not `pyunicorn`, which is heavy to install on Colab and sometimes fails); this is more transparent and easier to audit for the paper.


In [ ]:
# 1. Installation / imports
!pip install -q antropy 2>/dev/null

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy import stats
from itertools import groupby

plt.rcParams['figure.dpi'] = 110
np.random.seed(0)


## 1. Data loading

Two possible routes:

**Option A — Drive mounted** (recommended if the file is already there):


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Same convention as MFDFA_Pipeline.ipynb: the COMPLETE regional SPI-12 series
# (709 months, Dec-1965 to Dec-2024) lives at data/mfdfa/regional_SPI12.csv --
# NOT at data/final_training_csvs (those are the data already trimmed for
# training the neural networks, with lags consumed: 517 = 529 - 12 months).
DRIVE_BASE = "/content/drive/MyDrive/Proyecto_AI_SPI_2026"  # <-- edit if different
INPUT_CSV = f"{DRIVE_BASE}/data/mfdfa/regional_SPI12.csv"


In [ ]:
import os

REGIONS = ["Semi-arid", "Highlands", "Mountains", "Canyons"]
EXPECTED_N_MONTHS = 709      # December 1965 to December 2024
EXPECTED_START = "1965-12-01"
EXPECTED_END = "2024-12-01"

if not os.path.isfile(INPUT_CSV):
    print(f"ERROR: {INPUT_CSV} does not exist.")
    print("Check DRIVE_BASE, or whether regional_SPI12.csv has not yet been generated/uploaded.")
else:
    print(f"File found: {INPUT_CSV}")


In [ ]:
# Option B -- if regional_SPI12.csv is not yet on Drive, upload it directly to Colab:
# from google.colab import files
# uploaded = files.upload()  # select regional_SPI12.csv
# INPUT_CSV = list(uploaded.keys())[0]
# (then re-run the loader cell below)


In [ ]:
# --- Loader: same source and verification as Step 1 of MFDFA_Pipeline.ipynb ---
spi = pd.read_csv(INPUT_CSV, parse_dates=["date"])
spi = spi.sort_values("date").reset_index(drop=True)

print(f"Loaded {len(spi)} months of regional SPI-12 from: {INPUT_CSV}")
print(f"Date range: {spi['date'].min().date()} to {spi['date'].max().date()}")

if len(spi) != EXPECTED_N_MONTHS:
    print(f"\n  *** WARNING: expected {EXPECTED_N_MONTHS} months "
          f"({EXPECTED_START} to {EXPECTED_END}), found {len(spi)}. "
          f"Verify INPUT_CSV before proceeding. ***")
elif str(spi["date"].min().date()) != EXPECTED_START or str(spi["date"].max().date()) != EXPECTED_END:
    print(f"\n  *** WARNING: row count matches but the date range does not "
          f"({EXPECTED_START} to {EXPECTED_END} expected). Verify INPUT_CSV. ***")
else:
    print("  Row count and date range match Dec-1965 to Dec-2024. OK.")

if spi[REGIONS].isna().any().any():
    print("\n  *** WARNING: missing values found in one or more regional series. ***")
    print(spi[REGIONS].isna().sum())

df_regions = spi[REGIONS].copy()
df_regions.index = spi["date"]

print("\nDescriptive statistics:")
print(df_regions.describe().T[["min", "mean", "max", "std"]].round(3))
print()
print(df_regions.shape)
df_regions.head()


In [ ]:
# --- Duplicate check across regions ---
# If two columns end up identical (or nearly so), something was resolved incorrectly
# in the loader: check the 'Source summary per region' print from the previous cell
# to see which file/column was used in each case, and adjust REGION_ALIASES or the
# actual file/column names accordingly.

regions = list(df_regions.columns)
problem = False
for i in range(len(regions)):
    for j in range(i + 1, len(regions)):
        r1, r2 = regions[i], regions[j]
        if df_regions[r1].equals(df_regions[r2]):
            print(f'ERROR: {r1} and {r2} are EXACTLY equal. The loader resolved them to the same source.')
            problem = True
        else:
            corr = df_regions[r1].corr(df_regions[r2])
            if corr > 0.999:
                print(f'WARNING: {r1} and {r2} have correlation {corr:.4f} (suspiciously high, check it).')

if problem:
    raise ValueError('There are regions with identical series. Fix the loader before continuing '
                      '(check the source summary printed above).')
else:
    print('OK: the 4 regional series are distinct from each other.')


In [ ]:
# --- Time axis ---
# df_regions already carries the real date index (from the 'date' column of the CSV),
# taken directly in the loader cell -- no range needs to be assumed.
N = len(df_regions)

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for ax, region in zip(axes, df_regions.columns):
    ax.plot(df_regions.index, df_regions[region], lw=0.8)
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_ylabel(region, fontsize=9)
axes[-1].set_xlabel('Year')
fig.suptitle('Regional SPI-12 series (Zacatecas) — 709 months, Dec-1965 to Dec-2024')
plt.tight_layout()
plt.show()


## 2. Embedding parameter selection (τ, m)

Unlike the earlier attempt (dimension=1, delay=1, with no real embedding), here we select:

- **τ (delay)**: first local minimum of the Average Mutual Information (AMI).
- **m (embedding dimension)**: False Nearest Neighbors (FNN) method, looking for the value of *m* where the percentage of false neighbors drops below a threshold (e.g., 1–5%).


In [ ]:
def average_mutual_information(x, max_lag=24, bins=16):
    """AMI in bits for lags 0..max_lag using a 2D histogram."""
    x = np.asarray(x)
    ami = []
    for lag in range(1, max_lag + 1):
        x1, x2 = x[:-lag], x[lag:]
        c_xy, xedges, yedges = np.histogram2d(x1, x2, bins=bins)
        pxy = c_xy / c_xy.sum()
        px = pxy.sum(axis=1, keepdims=True)
        py = pxy.sum(axis=0, keepdims=True)
        with np.errstate(divide='ignore', invalid='ignore'):
            terms = pxy * np.log2(pxy / (px * py))
        terms = np.nan_to_num(terms)
        ami.append(terms.sum())
    return np.array(ami)


def first_minimum(series):
    """Index (1-based lag) of the first local minimum; if none, returns the global argmin."""
    for i in range(1, len(series) - 1):
        if series[i] < series[i - 1] and series[i] < series[i + 1]:
            return i + 1  # +1 because lag starts at 1
    return int(np.argmin(series)) + 1


tau_per_region = {}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, region in zip(axes, df_regions.columns):
    ami = average_mutual_information(df_regions[region].values, max_lag=24)
    tau = first_minimum(ami)
    tau_per_region[region] = tau
    ax.plot(range(1, 25), ami, marker='o', ms=3)
    ax.axvline(tau, color='red', ls='--', lw=1)
    ax.set_title(f'{region}\nτ = {tau}', fontsize=9)
    ax.set_xlabel('lag (months)')
plt.tight_layout()
plt.show()
print(tau_per_region)


In [ ]:
def false_nearest_neighbors(x, tau, max_dim=8, Rtol=15.0, Atol=2.0):
    """Percentage of false nearest neighbors for dimensions 1..max_dim."""
    x = np.asarray(x)
    N = len(x)
    Ra = np.std(x)
    fnn_pct = []
    for m in range(1, max_dim + 1):
        L = N - (m) * tau
        if L <= 10:
            fnn_pct.append(np.nan)
            continue
        Em = np.array([x[i:i + m * tau:tau] for i in range(L)])
        Em1 = np.array([x[i:i + (m + 1) * tau:tau] for i in range(L)])
        # nearest neighbor in dimension m
        Dm = squareform(pdist(Em))
        np.fill_diagonal(Dm, np.inf)
        nn_idx = np.argmin(Dm, axis=1)
        Rm = Dm[np.arange(L), nn_idx]
        extra = np.abs(Em1[:, -1] - Em1[nn_idx, -1])
        with np.errstate(divide='ignore', invalid='ignore'):
            crit1 = (extra / Rm) > Rtol
        crit2 = (np.sqrt(Rm ** 2 + extra ** 2) / Ra) > Atol
        false_pts = np.sum(crit1 | crit2)
        fnn_pct.append(100.0 * false_pts / L)
    return np.array(fnn_pct)


m_per_region = {}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, region in zip(axes, df_regions.columns):
    tau = tau_per_region[region]
    fnn = false_nearest_neighbors(df_regions[region].values, tau, max_dim=8)
    # first m where FNN% < 5%
    below = np.where(fnn < 5.0)[0]
    m_sel = int(below[0] + 1) if len(below) else int(np.nanargmin(fnn) + 1)
    m_per_region[region] = m_sel
    ax.plot(range(1, 9), fnn, marker='o', ms=3)
    ax.axhline(5, color='gray', ls=':', lw=1)
    ax.axvline(m_sel, color='red', ls='--', lw=1)
    ax.set_title(f'{region}\nm = {m_sel}', fontsize=9)
    ax.set_xlabel('dimension m')
    ax.set_ylabel('% FNN')
plt.tight_layout()
plt.show()
print(m_per_region)


## 3. Recurrence Plots with correct embedding and fixed-recurrence-rate threshold

The phase space is reconstructed with the (τ, m) selected per region, and the threshold ε is fixed so that the target recurrence rate (%REC) is constant (e.g., 5%) across regions — this way RQA comparisons between regions are not biased by arbitrary thresholds (unlike the fixed 1σ threshold used in the earlier attempt).

**Theiler window correction:** a band of width W around the main diagonal (offsets < W) is excluded from the recurrence matrix. Without this, neighboring time points are trivially "recurrent" to each other simply because SPI-12 (a 12-month moving average) changes slowly from one month to the next — this is not genuine recurrence to a distant past state, but an artifact first described by Theiler (1986) and explicitly flagged as a required correction in Marwan et al. (2007), the main methodological reference for this notebook. We use **W = τ** per region (the decorrelation timescale already estimated in Section 2) as a principled, non-arbitrary choice. This correction was added after finding that, without it, Mountains produced a spurious Lmax=343 driven entirely by an offset=1 diagonal (i.e., consecutive months trivially matching each other) — see §4.1.


In [ ]:
def embed(x, m, tau):
    N = len(x)
    L = N - (m - 1) * tau
    return np.array([x[i:i + m * tau:tau] for i in range(L)])


def theiler_mask(N, W):
    """Boolean mask, True where |i-j| < W (the band to EXCLUDE from RQA computations)."""
    i_idx, j_idx = np.indices((N, N))
    return np.abs(i_idx - j_idx) < W


def recurrence_matrix_target_RR(x, m, tau, target_RR=0.05, theiler_window=1):
    """Builds the binary recurrence matrix, adjusting epsilon to reach target_RR,
    excluding a Theiler window of half-width `theiler_window` around the main diagonal
    (both from the epsilon calibration and from the final matrix R)."""
    E = embed(np.asarray(x), m, tau)
    D = squareform(pdist(E, metric='euclidean'))
    N = D.shape[0]
    excl = theiler_mask(N, theiler_window)
    valid = ~excl
    eps = np.percentile(D[valid], target_RR * 100)
    R = ((D <= eps) & valid).astype(np.uint8)
    return R, eps, E, valid


RP = {}
EPS = {}
VALID_MASK = {}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, region in zip(axes, df_regions.columns):
    tau, m = tau_per_region[region], m_per_region[region]
    W = tau  # Theiler window = decorrelation time already estimated in Section 2
    R, eps, E, valid = recurrence_matrix_target_RR(df_regions[region].values, m, tau,
                                                     target_RR=0.05, theiler_window=W)
    RP[region] = R
    EPS[region] = eps
    VALID_MASK[region] = valid
    ax.imshow(R, cmap='Greys', origin='lower', interpolation='nearest')
    ax.set_title(f'{region} (τ={tau}, m={m}, W={W})', fontsize=9)
plt.tight_layout()
plt.show()


## 4. RQA measures

We compute, per region, from the distributions of diagonal and vertical line lengths:

- **RR** (recurrence rate)
- **DET** (determinism): fraction of recurrent points in diagonal lines of length ≥ Lmin
- **L** (mean diagonal line length)
- **Lmax**
- **ENTR** (Shannon entropy of diagonal line lengths)
- **LAM** (laminarity): fraction of points in vertical lines of length ≥ Vmin — measures "trapped" states
- **TT** (trapping time): mean vertical line length — typical duration of a regime before transitioning

This is the part that was missing in the earlier attempt: without these measures, the Recurrence Plot is just an image, not a quantifiable result.


In [ ]:
def diag_line_lengths(R, Lmin=2):
    N = R.shape[0]
    lengths = []
    # traverse all diagonals (offsets within the Theiler window are already zeroed out in R)
    for offset in range(1, N):
        diag = np.diagonal(R, offset=offset)
        for val, group in groupby(diag):
            if val == 1:
                run_len = len(list(group))
                if run_len >= Lmin:
                    lengths.append(run_len)
    return np.array(lengths)


def vert_line_lengths(R, Vmin=2):
    N = R.shape[0]
    lengths = []
    for col in range(N):
        colvals = R[:, col]
        for val, group in groupby(colvals):
            if val == 1:
                run_len = len(list(group))
                if run_len >= Vmin:
                    lengths.append(run_len)
    return np.array(lengths)


def rqa_measures(R, valid_mask, Lmin=2, Vmin=2):
    dlens = diag_line_lengths(R, Lmin)
    vlens = vert_line_lengths(R, Vmin)

    total_possible_points = valid_mask.sum()          # denominator for RR (Theiler band excluded)
    total_recurrent_points = R.sum()                    # diagonal/band already excluded, no need to subtract N

    RR = total_recurrent_points / total_possible_points if total_possible_points > 0 else np.nan
    DET = dlens.sum() / total_recurrent_points if total_recurrent_points > 0 and len(dlens) else np.nan
    Lmean = dlens.mean() if len(dlens) else np.nan
    Lmax = dlens.max() if len(dlens) else np.nan

    if len(dlens):
        vals, counts = np.unique(dlens, return_counts=True)
        p = counts / counts.sum()
        ENTR = -np.sum(p * np.log(p))
    else:
        ENTR = np.nan

    LAM = vlens.sum() / total_recurrent_points if total_recurrent_points > 0 and len(vlens) else np.nan
    TT = vlens.mean() if len(vlens) else np.nan

    return dict(RR=RR, DET=DET, L=Lmean, Lmax=Lmax, ENTR=ENTR, LAM=LAM, TT=TT)


rqa_table = {}
for region in df_regions.columns:
    rqa_table[region] = rqa_measures(RP[region], VALID_MASK[region], Lmin=2, Vmin=2)

rqa_df = pd.DataFrame(rqa_table).T
rqa_df


**How to read this table:**
- Regions with **higher %LAM and TT** tend to "stay" longer in a drought or wet regime before shifting — operational persistence.
- Regions with **higher %DET** have more deterministic/predictable dynamics; low %DET suggests a larger stochastic component.
- It is worth cross-checking this table against **h(2)** (equivalent to the classical DFA exponent) from your MF-DFA pipeline — they should be consistent: stronger persistence (h(2) > 0.5) ↔ higher LAM/TT.


### 4.1 Diagnostic: temporal location of Lmax (Mountains)

Historical note: an earlier run (without the Theiler window) produced `Lmax=343` for Mountains, driven entirely by an offset=1 diagonal — i.e., consecutive months trivially matching each other due to the smoothness of SPI-12, not genuine recurrence. That is exactly the artifact the Theiler-window correction in §3 addresses. Re-run this diagnostic on the corrected `RP` to confirm the new Lmax reflects genuine recurrence to a distinct past/future state (offset >= τ).


In [ ]:
def find_longest_diagonal(R, Lmin=2):
    """Returns (offset, start_row, length) of the longest diagonal line in R
    (excluding the identity line, offset=0)."""
    N = R.shape[0]
    best = (None, None, 0)
    for offset in range(1, N):
        diag = np.diagonal(R, offset=offset)
        pos = 0
        for val, group in groupby(diag):
            run_len = len(list(group))
            if val == 1 and run_len >= Lmin and run_len > best[2]:
                best = (offset, pos, run_len)
            pos += run_len
    return best  # (offset, start_row, length)


region = 'Mountains'
offset, start_row, length = find_longest_diagonal(RP[region])
print(f'{region}: longest diagonal line -> offset={offset}, start_row={start_row}, length={length}')

# The two time segments involved in this diagonal line:
seg_A_idx = np.arange(start_row, start_row + length)
seg_B_idx = np.arange(start_row + offset, start_row + offset + length)

dates_A = df_regions.index[seg_A_idx]
dates_B = df_regions.index[seg_B_idx]

print(f'\nSegment A: {dates_A.min().date()} to {dates_A.max().date()} ({length} months)')
print(f'Segment B: {dates_B.min().date()} to {dates_B.max().date()} ({length} months)')

vals_A = df_regions[region].values[seg_A_idx]
vals_B = df_regions[region].values[seg_B_idx]

print(f'\nSegment A -- mean: {vals_A.mean():.3f}, std: {vals_A.std():.3f}, min: {vals_A.min():.3f}, max: {vals_A.max():.3f}')
print(f'Segment B -- mean: {vals_B.mean():.3f}, std: {vals_B.std():.3f}, min: {vals_B.min():.3f}, max: {vals_B.max():.3f}')

# Artifact warning signs: near-zero variance, or exactly repeated values
n_unique_A = len(np.unique(vals_A))
n_unique_B = len(np.unique(vals_B))
print(f'\nUnique values in Segment A: {n_unique_A}/{length}  |  Segment B: {n_unique_B}/{length}')
if vals_A.std() < 0.05 or vals_B.std() < 0.05:
    print('*** WARNING: very low standard deviation in at least one segment -- possible artifact (near-flat series). ***')
if n_unique_A < length * 0.5 or n_unique_B < length * 0.5:
    print('*** WARNING: many repeated values -- possible imputation artifact or truncated data. ***')

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharey=True)
axes[0].plot(dates_A, vals_A, marker='o', ms=3, color='tab:blue')
axes[0].set_title(f'Segment A: {dates_A.min().date()} - {dates_A.max().date()}')
axes[0].axhline(0, color='gray', lw=0.5)
axes[1].plot(dates_B, vals_B, marker='o', ms=3, color='tab:orange')
axes[1].set_title(f'Segment B: {dates_B.min().date()} - {dates_B.max().date()}')
axes[1].axhline(0, color='gray', lw=0.5)
fig.suptitle(f'{region}: the two segments forming the longest diagonal line (Lmax={length})')
plt.tight_layout()
plt.show()

# Context: where these segments fall within the full Mountains series
fig2, ax2 = plt.subplots(figsize=(11, 3))
ax2.plot(df_regions.index, df_regions[region].values, lw=0.6, color='lightgray', zorder=1)
ax2.plot(dates_A, vals_A, color='tab:blue', lw=1.5, zorder=2, label='Segment A')
ax2.plot(dates_B, vals_B, color='tab:orange', lw=1.5, zorder=2, label='Segment B')
ax2.axhline(0, color='gray', lw=0.5)
ax2.legend()
ax2.set_title(f'{region}: location of Segments A and B within the full series')
plt.tight_layout()
plt.show()


# --- Objective check: are Segment A and Segment B actually similar, or anti-correlated? ---
# Two independent date axes (different calendar years) can create a visual illusion of
# "mirroring" even when the underlying pattern is genuinely similar -- or mask a real
# inversion. Check with a number, not just the eye.
pearson_r = np.corrcoef(vals_A, vals_B)[0, 1]
print(f'\nPearson correlation between Segment A and Segment B: r = {pearson_r:.3f}')
if pearson_r < -0.3:
    print('*** NOTE: negative correlation -- the two segments genuinely move in opposite')
    print('    directions. This would be unexpected for a recurrence-plot match (small')
    print('    Euclidean distance normally implies similar, not opposite, values) and is')
    print('    worth investigating further (e.g., a sign/indexing issue) rather than')
    print('    reported as-is. ***')
elif pearson_r > 0.3:
    print('Positive correlation: consistent with genuine recurrence (similar trajectories).')
else:
    print('Weak/near-zero correlation: the two segments are neither clearly similar nor')
    print('opposite in overall shape -- remember the RQA distance only constrains the')
    print(f'{m_per_region[region]} embedded (sparsely tau-spaced) coordinates, not necessarily')
    print('every consecutive month in between.')

# Overlay both segments on a common "months since segment start" axis, removing any
# illusion caused by the two panels having different absolute-date x-axes above.
fig3, ax3 = plt.subplots(figsize=(9, 4))
month_idx = np.arange(length)
ax3.plot(month_idx, vals_A, marker='o', ms=3, color='tab:blue', label=f'Segment A ({dates_A.min().date()})')
ax3.plot(month_idx, vals_B, marker='o', ms=3, color='tab:orange', label=f'Segment B ({dates_B.min().date()})')
ax3.axhline(0, color='gray', lw=0.5)
ax3.set_xlabel('Months since segment start')
ax3.set_ylabel('SPI-12')
ax3.set_title(f'{region}: Segment A vs Segment B, aligned by relative month (Pearson r={pearson_r:.3f})')
ax3.legend()
plt.tight_layout()
plt.show()


## 5. Cross-Recurrence between regions (synchronization of drought regimes)

Following Semeraro et al. (2020): we compute the **Cross Recurrence Plot (CRP)** between each pair of regions and the **F-synchronization measure** (density of points on the main diagonal of the CRP). F=1 indicates full synchronization; F→0, desynchronization.

Here the threshold is fixed so that both signals have the same RR (as in the reference paper), allowing comparability.


In [ ]:
from itertools import combinations

MIN_EMBEDDED_POINTS = 5  # minimum number of embedded vectors for F-recurrence to be meaningful

def cross_recurrence_F(x, y, m, tau, target_RR=0.05):
    Ex = embed(np.asarray(x), m, tau)
    Ey = embed(np.asarray(y), m, tau)
    L = min(len(Ex), len(Ey))
    if L < MIN_EMBEDDED_POINTS:
        # Window/series too short for (m, tau): (m-1)*tau consumes almost all the points.
        raise ValueError(
            f"Too few points after embedding (L={L} with m={m}, tau={tau}; "
            f"need >= {MIN_EMBEDDED_POINTS}). Input series has {len(x)} points; "
            f"(m-1)*tau={(m - 1) * tau} is eating up almost all of it."
        )
    Ex, Ey = Ex[:L], Ey[:L]
    Dxy = np.linalg.norm(Ex[:, None, :] - Ey[None, :, :], axis=-1)
    eps = np.percentile(Dxy, target_RR * 100)
    C = (Dxy <= eps).astype(np.uint8)
    F = np.trace(C) / L  # density on the main diagonal (line of synchronization)
    return F, C

pairs = list(combinations(df_regions.columns, 2))
F_results = {}
for r1, r2 in pairs:
    tau = int(np.mean([tau_per_region[r1], tau_per_region[r2]]))
    m = int(np.mean([m_per_region[r1], m_per_region[r2]]))
    F, C = cross_recurrence_F(df_regions[r1].values, df_regions[r2].values, m, tau, target_RR=0.05)
    F_results[(r1, r2)] = F

F_df = pd.Series(F_results).rename('F_recurrence').to_frame()
F_df


## 6. Sliding-window analysis: synchronization before/after documented droughts

We adapt the approach of Semeraro et al. (2020, section 3.2) — 1-year windows, comparing the F measure before/after events, with a Fisher test (normality) and Student's t-test (difference of means).

**Adjustment required:** the years marked below are placeholders with widely documented drought events in Mexico/Zacatecas (e.g., 2011, 2020–2022). Replace them with the years that your own domain knowledge (and your previous SPI publications) identify as severe drought episodes in the studied regions — this is exactly the kind of domain validation you already applied when discarding the multifractal clustering in the MF-Cluster-LNN project.


In [ ]:
# --- Embedding specific to short windows ---
# The (tau, m) estimated in Section 2 is optimal for the FULL series (709 months) and
# turns out to be too "heavy" for windows spanning just a few years: (m-1)*tau can
# consume 27-40 months just to "warm up" the embedding, leaving zero usable vectors in a
# 24-36 month window. Therefore, following the practice of Semeraro et al. (2020) for
# their windowed synchronization analysis, we use here a minimal, fixed embedding,
# independent of the global (tau, m) -- appropriate for capturing short-term structure
# without eating up the window.
WINDOW_TAU = 1
WINDOW_M = 2

DROUGHT_EVENTS = [1996, 2011, 2012, 2021, 2023]  # documented severe droughts (see documentation cell below)

def sliding_F_before_after(x, y, m, tau, event_year, date_index, window_years=3, target_RR=0.05,
                            step_months=3, win_months=12, verbose=False):
    before_mask = (date_index.year >= event_year - window_years) & (date_index.year < event_year)
    after_mask = (date_index.year > event_year) & (date_index.year <= event_year + window_years)

    def sub_F(mask):
        idx_ = np.where(mask)[0]
        if len(idx_) < win_months:
            return []
        vals = []
        skipped = 0
        for start in range(0, len(idx_) - win_months + 1, step_months):
            sl = idx_[start:start + win_months]
            try:
                F, _ = cross_recurrence_F(x[sl], y[sl], m, tau, target_RR)
                vals.append(F)
            except (ValueError, IndexError):
                skipped += 1
                continue
        if verbose and skipped:
            print(f'    ({skipped} sub-window(s) skipped: not enough points for m={m}, tau={tau})')
        return vals

    before_vals = sub_F(before_mask)
    after_vals = sub_F(after_mask)
    if verbose:
        print(f'  event {event_year}: {before_mask.sum()} months before -> {len(before_vals)} windows, '
              f'{after_mask.sum()} months after -> {len(after_vals)} windows')
    return before_vals, after_vals


rows = []
for r1, r2 in pairs:
    # NOTE: here we use WINDOW_M/WINDOW_TAU (fixed, lightweight embedding), NOT the global
    # (m, tau) from tau_per_region/m_per_region -- that one is for Sections 4-5 (full series).
    for event in DROUGHT_EVENTS:
        before, after = sliding_F_before_after(
            df_regions[r1].values, df_regions[r2].values, WINDOW_M, WINDOW_TAU, event, df_regions.index,
            verbose=True
        )
        if len(before) < 2 or len(after) < 2:
            print(f'  -> {r1}-{r2}, event {event}: not enough windows for a statistical test, skipping.')
            continue
        fisher_p = stats.levene(before, after).pvalue  # variance homogeneity (robust proxy)
        ttest_p = stats.ttest_ind(before, after, equal_var=False).pvalue
        rows.append(dict(
            pair=f'{r1}-{r2}', event=event,
            n_before=len(before), n_after=len(after),
            mean_before=np.mean(before), mean_after=np.mean(after),
            std_before=np.std(before), std_after=np.std(after),
            levene_p=fisher_p, ttest_p=ttest_p
        ))

sliding_df = pd.DataFrame(rows)

if not sliding_df.empty:
    # --- Multiple-comparison correction ---
    # With 6 pairs x up to 5 events = up to 24-30 independent tests, at alpha=0.05 we
    # expect ~1 false positive by chance alone. We apply Bonferroni (conservative) and
    # Benjamini-Hochberg/FDR (less conservative, more standard for exploratory work) to
    # the t-test p-values.
    n_tests = len(sliding_df)
    alpha = 0.05

    # Bonferroni
    sliding_df['ttest_p_bonferroni'] = (sliding_df['ttest_p'] * n_tests).clip(upper=1.0)
    sliding_df['sig_bonferroni'] = sliding_df['ttest_p_bonferroni'] < alpha

    # Benjamini-Hochberg (FDR)
    order = np.argsort(sliding_df['ttest_p'].values)
    ranked_p = sliding_df['ttest_p'].values[order]
    bh_crit = (np.arange(1, n_tests + 1) / n_tests) * alpha
    passed = ranked_p <= bh_crit
    # the largest rank that passes sets the effective threshold; all lower ranks are also accepted
    if passed.any():
        max_rank_passed = np.max(np.where(passed)[0])
        bh_significant_sorted = np.zeros(n_tests, dtype=bool)
        bh_significant_sorted[:max_rank_passed + 1] = True
    else:
        bh_significant_sorted = np.zeros(n_tests, dtype=bool)
    sig_fdr = np.zeros(n_tests, dtype=bool)
    sig_fdr[order] = bh_significant_sorted
    sliding_df['sig_fdr_bh'] = sig_fdr

    n_sig_raw = (sliding_df['ttest_p'] < alpha).sum()
    n_sig_bonf = sliding_df['sig_bonferroni'].sum()
    n_sig_fdr = sliding_df['sig_fdr_bh'].sum()
    print(f'Total tests: {n_tests}')
    print(f'  Significant, uncorrected (p<0.05): {n_sig_raw}')
    print(f'  Significant after Bonferroni:       {n_sig_bonf}')
    print(f'  Significant after FDR (Benjamini-Hochberg): {n_sig_fdr}')
    sliding_df = sliding_df.sort_values('ttest_p').reset_index(drop=True)
else:
    print()
    print('WARNING: sliding_df is still empty. Check the diagnostic messages printed above.')

sliding_df


**Interpretation:** a `ttest_p < 0.05` indicates that synchronization between that pair of regions changed significantly around the drought event — analogous to the disruption Semeraro et al. found in Amazonian LST during the 2015 El Niño. If no `ttest_p` is significant, it is evidence (as in the reference paper, where AMO did not affect EVI-NDWI synchronization) that Zacatecas' regions respond in a coupled way even under severe drought — an interesting finding in itself, whether or not it agrees with the official regionalization validated in your MF-DFA manuscript.


### Documentation of the drought events used in `DROUGHT_EVENTS`

For traceability when writing methods/discussion — these are not placeholders, they are source-backed.

**National sources (CONAGUA via SMN, and media reproducing official data):**

- **1996** — considered the worst drought of the 1990s at the national level. Paralyzed agricultural exports from northern Mexico; ranchers forced to undersell or mass-slaughter entire herds. **Systemic/national** event.
- **2011** — record affected area since satellite records and modern measurements began; 95.12% of national territory showed some degree of impact by April 30, 2011, driven by La Niña. **Systemic/national event, continental scale**.
- **2021–2022** — widespread drought linked to North American water scarcity; >85% of the country under critical conditions in April 2021; by 2022 it led to urban water-shortage crises in northern states (e.g., Nuevo León). **Continental/national** scale, but with a **heterogeneous response within Zacatecas** (see below) — relevant: RQA captured that sub-state heterogeneity even though the forcing was national.
- **2023–2024** — 76% of national territory in moderate-to-exceptional (D4) drought; reservoir levels in the center-north of the country at historic lows before recovering.
  Sources: gob.mx/SMN (official statement), TV Azteca (reproduces CONAGUA/SMN data), El Informador, Wikipedia (Water scarcity in Mexico), Ganadería.com.

**Zacatecas-specific sources:**

- **2011–2012** — SciELO (worst in 70 years, ~13-month dry spell, >75,000 head of livestock lost in the state): S2007-24222016000500077, S2007-24222018000300047; Aliat Universidades — Conexxion (economic impact, Jan-Feb 2012).
- **2021** — direct grower testimony: critical year for ranching, concentrated in Sombrerete, Valparaíso, Río Grande (Highlands proper / Highlands-Mountains border); watering troughs and small dam reservoirs nearly dried up completely.
- **2023** — CONAGUA: deficits of up to 73% below average in certain months; 50 of 58 Zacatecas municipalities in extreme drought. **Not statistically testable** with the current windowing scheme (`window_years=3` requires data through ~2026; the series ends Dec-2024). Report as a limitation, do not silently omit.

**Emerging pattern (with the `sliding_df` results already corrected for multiple comparisons):**

- Systemic/national events with a homogeneous impact within the state (2011–2012) → *synchronization* between normally decoupled pairs (Semi-arid–Highlands, p=2.3×10⁻⁸, survives Bonferroni).
- The 2021 event, although continental in scale (>85% of the country), had a **regionally uneven** signature within Zacatecas: synchronization in Highlands–Canyons (p=0.0025, survives FDR) but desynchronization in Semi-arid–Highlands (p=0.008, confirmatory given the pre-specified field testimony, survives FDR).
- 1996 stands as an additional test of the "systemic event → synchronization" pattern, with the advantage of having abundant data before and after (unlike 2023).

This systemic-national-but-heterogeneous-within-state vs. regionally-concentrated-event distinction is a hypothesis to develop in the discussion, not a closed statistical conclusion.


## 7. Suggested next steps for the paper

1. Confirm/adjust `DROUGHT_EVENTS` with severe drought years documented specifically for each region of Zacatecas (check your own 2019/2022/2024 publications for already-validated dates).
2. Decide whether %LAM/TT should also be reported by sub-period (e.g., pre/post-2000) to check whether drought persistence has changed with climate change — this links directly to the moving-window DFA finding from the EUR-OPA report reviewed earlier.
3. Quantitatively cross-reference RQA vs. MF-DFA: is the region with the lowest multifractality (Canyons, per your manuscript under review) also the one with the lowest %DET/highest apparent randomness?
4. Consider journal choice: since RQA + drought is a relatively unexplored niche (the reference paper is about vegetation, not SPI directly), *Stochastic Environmental Research and Risk Assessment* or *Nonlinear Processes in Geophysics* might be a better fit than *Atmosphere*, to clearly differentiate it from the manuscript already in progress.
